# Desafio 1 - Gurmendi Alan

## Consigna

**Cada experimento realizado debe estar acompañado de una explicación o interpretación de lo observado.**

**1**. Vectorizar documentos. Tomar 5 documentos al azar y medir similaridad con el resto de los documentos.
Estudiar los 5 documentos más similares de cada uno analizar si tiene sentido
la similaridad según el contenido del texto y la etiqueta de clasificación.

**2**. Construir un modelo de clasificación por prototipos (tipo zero-shot). Clasificar los documentos de un conjunto de test comparando cada uno con todos los de entrenamiento y asignar la clase al label del documento del conjunto de entrenamiento con mayor similaridad.

**3**. Entrenar modelos de clasificación Naïve Bayes para maximizar el desempeño de clasificación
(f1-score macro) en el conjunto de datos de test. Considerar cambiar parámteros
de instanciación del vectorizador y los modelos y probar modelos de Naïve Bayes Multinomial
y ComplementNB.

**NO cambiar el hiperparámetro ngram_range de los vectorizadores**.

**4**. Transponer la matriz documento-término. De esa manera se obtiene una matriz
término-documento que puede ser interpretada como una colección de vectorización de palabras.
Estudiar ahora similaridad entre palabras tomando 5 palabras y estudiando sus 5 más similares.

**Elegir las palabras MANUALMENTE para evitar la aparición de términos poco interpretables**.


----

Realizamos los importes necesarios

In [82]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.metrics import f1_score
from sklearn.datasets import fetch_20newsgroups
import numpy as np 

Se cargan los datos que a utilizar

In [83]:
# cargamos los datos (ya separados de forma predeterminada en train y test)
newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
newsgroups_test = fetch_20newsgroups(subset='test', remove=('headers', 'footers', 'quotes'))

Se instancia el vectorizador y se arman los set de entrenamiento y test segun lo visto en clase

In [84]:
# instanciamos un vectorizador
tfidfvect = TfidfVectorizer()

X_train = tfidfvect.fit_transform(newsgroups_train.data)

idx2word = {v: k for k,v in tfidfvect.vocabulary_.items()}

y_train = newsgroups_train.target
X_test = tfidfvect.transform(newsgroups_test.data)
y_test = newsgroups_test.target

## Ejercicio 1

Se implementa una función que aplica cosine similarity entre un documento seleccionado y todos los documentos del conjunto de entrenamiento, mostrando los textos más similares junto con sus etiquetas y valores de similitud.

In [85]:
def obtener_documentos_similares(idx, cant_similares=5):
    print(f"Documento con id {idx}:\n")
    print({newsgroups_train.data[idx]})
    print(f"Label: {newsgroups_train.target_names[y_train[idx]]}\n")

    cossim = cosine_similarity(X_train[idx], X_train)[0]
    mostsim = np.argsort(cossim)[::-1][1:6]

    print("Documentos similares:")
    for i in mostsim:
        print(f"-------------------------- idx={i} --------------------------\n")
        print(f"{newsgroups_train.data[y_train[i]]}\n")
        print(f"Label: {newsgroups_train.target_names[y_train[i]]}")
        print(f"Similitud del coseno: {cossim[i]:.4f}\n")


In [86]:
obtener_documentos_similares(5129)

Documento con id 5129:

{"\nI just put one in my machine last week.  I have an AST 486/66.  I was\ngetting ~10million winmarks with my Diamond SS24, and the #9 board is\ndoing ~20million winmarks.  From my brief experiences with it, i'm very\nsatisfied.  BTW, this is with Win 3.1."}
Label: comp.os.ms-windows.misc

Documentos similares:
-------------------------- idx=6720 --------------------------


Do you have Weitek's address/phone number?  I'd like to get some information
about this chip.


Label: comp.sys.ibm.pc.hardware
Similitud del coseno: 0.2329

-------------------------- idx=2457 --------------------------


Do you have Weitek's address/phone number?  I'd like to get some information
about this chip.


Label: comp.sys.ibm.pc.hardware
Similitud del coseno: 0.2286

-------------------------- idx=1058 --------------------------

well folks, my mac plus finally gave up the ghost this weekend after
starting life as a 512k way back in 1985.  sooo, i'm in the market for a
new machin

El documento elegido pertenece a la temática de la computación. Si bien los documentos con mayor similitud de coseno no comparten la misma etiqueta, sí coinciden en la temática general, lo que muestra que el modelo TF-IDF agrupa textos relacionados por vocabulario técnico común.

In [87]:
obtener_documentos_similares(10)

Documento con id 10:

{'I have a line on a Ducati 900GTS 1978 model with 17k on the clock.  Runs\nvery well, paint is the bronze/brown/orange faded out, leaks a bit of oil\nand pops out of 1st with hard accel.  The shop will fix trans and oil \nleak.  They sold the bike to the 1 and only owner.  They want $3495, and\nI am thinking more like $3K.  Any opinions out there?  Please email me.\nThanks.  It would be a nice stable mate to the Beemer.  Then I\'ll get\na jap bike and call myself Axis Motors!\n\n-- \n-----------------------------------------------------------------------\n"Tuba" (Irwin)      "I honk therefore I am"     CompuTrac-Richardson,Tx\nirwin@cmptrc.lonestar.org    DoD #0826          (R75/6)'}
Label: rec.motorcycles

Documentos similares:
-------------------------- idx=3543 --------------------------

I have win 3.0 and downloaded several icons and BMP's but I can't figure out
how to change the "wallpaper" or use the icons.  Any help would be appreciated.


Thanx,

-Brando

In [88]:
obtener_documentos_similares(1820)

Documento con id 1820:

{"\nUSUALLY....go enough places and you'll see stuff happen you didn't think did.\n"}
Label: rec.autos

Documentos similares:
-------------------------- idx=5410 --------------------------

I have win 3.0 and downloaded several icons and BMP's but I can't figure out
how to change the "wallpaper" or use the icons.  Any help would be appreciated.


Thanx,

-Brando

Label: rec.motorcycles
Similitud del coseno: 0.2692

-------------------------- idx=11155 --------------------------

I recently posted an article asking what kind of rates single, male
drivers under 25 yrs old were paying on performance cars. Here's a summary of
the replies I received.
 
 
 
 
-------------------------------------------------------------------------------
 
I'm not under 25 anymore (but is 27 close enough).
 
1992 Dodge Stealth RT/Twin Turbo (300hp model).
No tickets, no accidents, own a house, have taken defensive driving 1,
airbag, abs, security alarm, single.
 
$1500/year  $500 decu

En este caso, el documento elegido resulta algo vago en la representación de la temática indicada por su etiqueta. Los documentos más similares tampoco pertenecen a la misma categoría, y la relación temática no es clara. Esto muestra que el modelo TF-IDF puede agrupar textos por coincidencias léxicas sin reflejar necesariamente una conexión semántica fuerte.

In [89]:
obtener_documentos_similares(11000)

Documento con id 11000:

{'\n\n\n\nJust out of curiousity, how old is Worden?\n--\n_______________________________________________________________________________'}
Label: sci.space

Documentos similares:
-------------------------- idx=2699 --------------------------

well folks, my mac plus finally gave up the ghost this weekend after
starting life as a 512k way back in 1985.  sooo, i'm in the market for a
new machine a bit sooner than i intended to be...

i'm looking into picking up a powerbook 160 or maybe 180 and have a bunch
of questions that (hopefully) somebody can answer:

* does anybody know any dirt on when the next round of powerbook
introductions are expected?  i'd heard the 185c was supposed to make an
appearence "this summer" but haven't heard anymore on it - and since i
don't have access to macleak, i was wondering if anybody out there had
more info...

* has anybody heard rumors about price drops to the powerbook line like the
ones the duo's just went through recently?


El documento elegido tiene un contenido muy breve y poco informativo respecto de su etiqueta. En consecuencia, los textos más similares pertenecen a categorías diversas y no guardan una relación temática clara, lo que refleja la limitación del modelo TF-IDF ante documentos con escaso contenido léxico.

In [90]:
obtener_documentos_similares(160)

Documento con id 160:

{'Mr. water-head,\ni never said that israel diverted lebanese rivers, in fact i said that\nisrael went into southern lebanon to  make sure that no \nwater is being used on the lebanese\nside, so that all water would run into Jordan river where there\nisrael will use it  !#$%^%&&*-head.'}
Label: talk.politics.mideast

Documentos similares:
-------------------------- idx=5294 --------------------------

I recently posted an article asking what kind of rates single, male
drivers under 25 yrs old were paying on performance cars. Here's a summary of
the replies I received.
 
 
 
 
-------------------------------------------------------------------------------
 
I'm not under 25 anymore (but is 27 close enough).
 
1992 Dodge Stealth RT/Twin Turbo (300hp model).
No tickets, no accidents, own a house, have taken defensive driving 1,
airbag, abs, security alarm, single.
 
$1500/year  $500 decut. State Farm Insurance (this includes the additional $100
for the $1,000,000 um

El documento elegido presenta un contenido claramente vinculado a la política en Medio Oriente, pero los textos más similares no comparten esa temática. Esto muestra que el modelo TF-IDF puede fallar cuando el vocabulario es limitado o poco representativo del contexto político del documento.

## Ejercicio 2

Se implementa una función de clasificación basada en similitud del coseno, que asigna a cada documento de prueba la etiqueta del documento de entrenamiento más similar

In [91]:
def predictor_por_similitud(X_train, y_train, X_test, y_test):
    

    sim = cosine_similarity(X_test, X_train)

    # Índice del vecino más similar por fila
    idx_mas_similar = np.argmax(sim, axis=1)

    # Predicción: etiqueta del vecino más similar
    y_pred = y_train[idx_mas_similar]

    # Evaluación
    f1_macro = f1_score(y_test, y_pred, average='macro')

    return y_pred, f1_macro

In [92]:
predictor_por_similitud(X_train, y_train, X_test, y_test)

(array([ 0, 19, 17, ..., 17, 12, 15], shape=(7532,)), 0.5049911553681621)

La función devuelve las etiquetas predichas para el conjunto de prueba y el valor del F1-score macro. En este caso, se obtuvo un F1 ≈ 0.50, lo que indica un desempeño moderado. El resultado muestra que el método basado únicamente en similitud del coseno logra captar cierta coherencia temática, aunque sin alcanzar la precisión de un modelo entrenado como Naive Bayes

## Ejercicio 3

Se armaron dos funciones para probar los modelos MultinomialNB y ComplementNB, permitiendo modificar el valor de alpha y decidir si se eliminan o no las stopwords mediante el parámetro correspondiente del vectorizador.

In [93]:
def test_multinomial(train_data, test_data, alpha_value=1, filter_stop_words=True):

    if filter_stop_words:
        tfidfvect = TfidfVectorizer(
        stop_words='english'
        )
        
    else:
        tfidfvect = TfidfVectorizer()


    X_train = tfidfvect.fit_transform(train_data)
    X_test  = tfidfvect.transform(test_data)


    # Instanciamos el modelo y lo entrenamos
    clf = MultinomialNB(alpha=alpha_value)
    clf.fit(X_train, y_train)

    # Obtenemos los resultados
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    if filter_stop_words:
        print(f"F1 macro para alpha {alpha_value} y sin stopwords: {f1:.4f}")
    else:
        print(f"F1 macro para alpha {alpha_value} y con stopwords: {f1:.4f}")

In [94]:
def test_complementNB(train_data, test_data, alpha_value=1, filter_stop_words=True):

    if filter_stop_words:
        tfidfvect = TfidfVectorizer(
        stop_words='english'
        )
        
    else:
        tfidfvect = TfidfVectorizer()


    X_train = tfidfvect.fit_transform(train_data)
    X_test  = tfidfvect.transform(test_data)


    # Instanciamos el modelo y lo entrenamos
    clf = ComplementNB(alpha=alpha_value)
    clf.fit(X_train, y_train)

    # Obtenemos los resultados
    y_pred = clf.predict(X_test)
    f1 = f1_score(y_test, y_pred, average='macro')
    if filter_stop_words:
        print(f"F1 macro para alpha {alpha_value} y sin stopwords: {f1:.4f}")
    else:
        print(f"F1 macro para alpha {alpha_value} y con stopwords: {f1:.4f}")

Se realizan pruebas sobre el modelo multinomial

In [95]:
test_multinomial(newsgroups_train.data, newsgroups_test.data, 1, True)
test_multinomial(newsgroups_train.data, newsgroups_test.data, 1, False)
test_multinomial(newsgroups_train.data, newsgroups_test.data, 0.5, True)
test_multinomial(newsgroups_train.data, newsgroups_test.data, 0.5, False)
test_multinomial(newsgroups_train.data, newsgroups_test.data, 0.1, True)
test_multinomial(newsgroups_train.data, newsgroups_test.data, 0.1, False)
test_multinomial(newsgroups_train.data, newsgroups_test.data, 0.01, True)
test_multinomial(newsgroups_train.data, newsgroups_test.data, 0.01, False)


F1 macro para alpha 1 y sin stopwords: 0.6468
F1 macro para alpha 1 y con stopwords: 0.5854
F1 macro para alpha 0.5 y sin stopwords: 0.6584
F1 macro para alpha 0.5 y con stopwords: 0.6153
F1 macro para alpha 0.1 y sin stopwords: 0.6726
F1 macro para alpha 0.1 y con stopwords: 0.6565
F1 macro para alpha 0.01 y sin stopwords: 0.6844
F1 macro para alpha 0.01 y con stopwords: 0.6829


Se observa que los modelos en los que se eliminan las stopwords del texto presentan un mejor desempeño en comparación con aquellos donde se mantienen. También se ve una mejora al disminuir el valor de α, y que la diferencia de rendimiento entre eliminar o no las stopwords se reduce a medida que α es menor.

Se realizan pruebas sobre el modelo omplementNB

In [96]:
test_complementNB(newsgroups_train.data, newsgroups_test.data, 1, True)
test_complementNB(newsgroups_train.data, newsgroups_test.data, 1, False)
test_complementNB(newsgroups_train.data, newsgroups_test.data, 0.5, True)
test_complementNB(newsgroups_train.data, newsgroups_test.data, 0.5, False)
test_complementNB(newsgroups_train.data, newsgroups_test.data, 0.1, True)
test_complementNB(newsgroups_train.data, newsgroups_test.data, 0.1, False)
test_complementNB(newsgroups_train.data, newsgroups_test.data, 0.01, True)
test_complementNB(newsgroups_train.data, newsgroups_test.data, 0.01, False)

F1 macro para alpha 1 y sin stopwords: 0.6936
F1 macro para alpha 1 y con stopwords: 0.6930
F1 macro para alpha 0.5 y sin stopwords: 0.6978
F1 macro para alpha 0.5 y con stopwords: 0.6961
F1 macro para alpha 0.1 y sin stopwords: 0.6919
F1 macro para alpha 0.1 y con stopwords: 0.6954
F1 macro para alpha 0.01 y sin stopwords: 0.6652
F1 macro para alpha 0.01 y con stopwords: 0.6689


En este caso, la inclusión o eliminación de las stopwords no muestra un impacto significativo en el desempeño del modelo. Se observa una leve mejora al disminuir el valor de α hasta cierto punto, aunque con valores demasiado bajos el rendimiento empeora ligeramente.

## Ejercicio 4

Arramos la matriz traspuesta

In [97]:
X_terms = X_train.T

In [98]:
vocab = tfidfvect.vocabulary_

In [99]:
idx2word = {v: k for k, v in vocab.items()}

In [100]:
vocab

{'was': 95844,
 'wondering': 97181,
 'if': 48754,
 'anyone': 18915,
 'out': 68847,
 'there': 88638,
 'could': 30074,
 'enlighten': 37335,
 'me': 60560,
 'on': 68080,
 'this': 88767,
 'car': 25775,
 'saw': 80623,
 'the': 88532,
 'other': 68781,
 'day': 31990,
 'it': 51326,
 'door': 34809,
 'sports': 84538,
 'looked': 57390,
 'to': 89360,
 'be': 21987,
 'from': 41715,
 'late': 55746,
 '60s': 9843,
 'early': 35974,
 '70s': 11174,
 'called': 25492,
 'bricklin': 24160,
 'doors': 34810,
 'were': 96247,
 'really': 76471,
 'small': 83426,
 'in': 49447,
 'addition': 16809,
 'front': 41724,
 'bumper': 24635,
 'separate': 81658,
 'rest': 77878,
 'of': 67670,
 'body': 23480,
 'is': 51136,
 'all': 17936,
 'know': 54632,
 'can': 25590,
 'tellme': 88143,
 'model': 62746,
 'name': 64931,
 'engine': 37287,
 'specs': 84276,
 'years': 99911,
 'production': 73373,
 'where': 96433,
 'made': 59079,
 'history': 46814,
 'or': 68409,
 'whatever': 96395,
 'info': 49932,
 'you': 100208,
 'have': 45885,
 'funky':

Se implementa una función que calcula la similitud del coseno entre palabras, a partir de la matriz término–documento. Dada una palabra, muestra las más similares según su patrón de coocurrencia en los documentos del conjunto de entrenamiento.

In [101]:
def obtener_palabras_similares(palabra, cant_similares=5):
    if palabra not in vocab:
        print(f"La palabra '{palabra}' no está en el vocabulario.")
        return

    idx = vocab[palabra]
    print(f"Palabra con id {idx}:\n")
    print({palabra})
    print(f"ID término: {idx}\n")

    cossim = cosine_similarity(X_terms[idx], X_terms)[0]
    orden = np.argsort(cossim)[::-1]
    orden = orden[orden != idx]
    mostsim = orden[:cant_similares]

    print("Palabras similares:")
    for i in mostsim:
        print(f"-------------------------- idx={i} --------------------------\n")
        print(f"{idx2word[i]}\n")
        print(f"Similitud del coseno: {cossim[i]:.4f}\n")

In [102]:
obtener_palabras_similares('windows')

Palabra con id 96760:

{'windows'}
ID término: 96760

Palabras similares:
-------------------------- idx=34844 --------------------------

dos

Similitud del coseno: 0.3037

-------------------------- idx=63693 --------------------------

ms

Similitud del coseno: 0.2320

-------------------------- idx=61614 --------------------------

microsoft

Similitud del coseno: 0.2219

-------------------------- idx=66781 --------------------------

nt

Similitud del coseno: 0.2140

-------------------------- idx=41127 --------------------------

for

Similitud del coseno: 0.1930



Para la palabra “windows”, las más similares corresponden a términos relacionados con sistemas operativos y software. Esto muestra que la representación TF-IDF captura correctamente la relación temática dentro del dominio informático.

In [103]:
obtener_palabras_similares('engine')

Palabra con id 37287:

{'engine'}
ID término: 37287

Palabras similares:
-------------------------- idx=44750 --------------------------

guesser

Similitud del coseno: 0.2015

-------------------------- idx=90269 --------------------------

tripmeter

Similitud del coseno: 0.2015

-------------------------- idx=63306 --------------------------

mountings

Similitud del coseno: 0.2015

-------------------------- idx=76559 --------------------------

rebuilt

Similitud del coseno: 0.2010

-------------------------- idx=29685 --------------------------

conver

Similitud del coseno: 0.2005



En el caso de la palabra “engine”, las más similares no presentan una relación semántica clara, lo que sugiere que el término aparece en contextos variados o con poca frecuencia.

In [104]:
obtener_palabras_similares('jesus')

Palabra con id 52157:

{'jesus'}
ID término: 52157

Palabras similares:
-------------------------- idx=27196 --------------------------

christ

Similitud del coseno: 0.3039

-------------------------- idx=43842 --------------------------

god

Similitud del coseno: 0.2688

-------------------------- idx=54287 --------------------------

kingdom

Similitud del coseno: 0.2130

-------------------------- idx=59895 --------------------------

mat

Similitud del coseno: 0.1967

-------------------------- idx=22698 --------------------------

bible

Similitud del coseno: 0.1951



Para la palabra “jesus”, las más similares pertenecen claramente al mismo campo semántico religioso

In [105]:
obtener_palabras_similares('car')

Palabra con id 25775:

{'car'}
ID término: 25775

Palabras similares:
-------------------------- idx=25921 --------------------------

cars

Similitud del coseno: 0.1797

-------------------------- idx=30533 --------------------------

criterium

Similitud del coseno: 0.1770

-------------------------- idx=27562 --------------------------

civic

Similitud del coseno: 0.1748

-------------------------- idx=69218 --------------------------

owner

Similitud del coseno: 0.1689

-------------------------- idx=32150 --------------------------

dealer

Similitud del coseno: 0.1681



Para la palabra “car”, las más similares —cars, dealer, civic y owner— están vinculadas al ámbito automotriz, mostrando que el modelo TF-IDF identifica correctamente relaciones léxicas dentro de una misma temática.

In [106]:
obtener_palabras_similares('ghost')

Palabra con id 43346:

{'ghost'}
ID término: 43346

Palabras similares:
-------------------------- idx=86570 --------------------------

suptermarket

Similitud del coseno: 0.5594

-------------------------- idx=45875 --------------------------

haunts

Similitud del coseno: 0.5579

-------------------------- idx=89576 --------------------------

tore

Similitud del coseno: 0.4986

-------------------------- idx=96252 --------------------------

werewolf

Similitud del coseno: 0.4724

-------------------------- idx=86461 --------------------------

superman

Similitud del coseno: 0.4427



En el caso de la palabra “ghost”, algunas palabras similares (como haunts o werewolf) guardan coherencia temática, mientras que otras no presentan una relación clara.